In [2]:
import math
import random
from copy import deepcopy


# =========================================
# TIC TAC TOE
# =========================================

class TicTacToe:

    def __init__(self):

        self.board = [
            ['X', 'O', 'X'],
            ['O', 'X', ' '],
            [' ', ' ', 'O']
        ]

    def print_board(self):

        for row in self.board:
            print(row)

    def available_moves(self):

        moves = []

        for i in range(3):
            for j in range(3):

                if self.board[i][j] == ' ':
                    moves.append((i, j))

        return moves

    def make_move(self, move, player):

        i, j = move
        self.board[i][j] = player

    def winner(self):

        lines = []

        # Rows
        lines.extend(self.board)

        # Columns
        for col in range(3):
            lines.append(
                [self.board[row][col] for row in range(3)]
            )

        # Diagonals
        lines.append(
            [self.board[i][i] for i in range(3)]
        )

        lines.append(
            [self.board[i][2 - i] for i in range(3)]
        )

        for line in lines:

            if line == ['X', 'X', 'X']:
                return 'X'

            if line == ['O', 'O', 'O']:
                return 'O'

        if len(self.available_moves()) == 0:
            return 'Draw'

        return None


# =========================================
# NODE CLASS
# =========================================

class Node:

    def __init__(self,
                 game,
                 parent=None,
                 move=None,
                 player='X'):

        self.game = deepcopy(game)

        self.parent = parent
        self.move = move
        self.player = player

        self.children = []

        self.visits = 0
        self.wins = 0

    def best_child(self):

        best_score = -math.inf
        best_node = None

        for child in self.children:

            # Prevent divide by zero
            if child.visits == 0:
                return child

            ucb = (
                (child.wins / child.visits)
                +
                1.41 * math.sqrt(
                    math.log(self.visits) / child.visits
                )
            )

            if ucb > best_score:
                best_score = ucb
                best_node = child

        return best_node


# =========================================
# RANDOM SIMULATION
# =========================================

def simulate(game, current_player):

    game_copy = deepcopy(game)

    while game_copy.winner() is None:

        move = random.choice(
            game_copy.available_moves()
        )

        game_copy.make_move(move, current_player)

        current_player = (
            'O' if current_player == 'X'
            else 'X'
        )

    return game_copy.winner()


# =========================================
# MONTE CARLO TREE SEARCH
# =========================================

def mcts(game, iterations=500):

    root = Node(game)

    for _ in range(iterations):

        node = root

        # ---------------------
        # Selection
        # ---------------------

        while node.children:
            node = node.best_child()

        # ---------------------
        # Expansion
        # ---------------------

        if node.game.winner() is None:

            for move in node.game.available_moves():

                new_game = deepcopy(node.game)

                new_game.make_move(
                    move,
                    node.player
                )

                next_player = (
                    'O' if node.player == 'X'
                    else 'X'
                )

                child = Node(
                    new_game,
                    node,
                    move,
                    next_player
                )

                node.children.append(child)

        # Choose random child
        if node.children:
            node = random.choice(node.children)

        # ---------------------
        # Simulation
        # ---------------------

        result = simulate(
            node.game,
            node.player
        )

        # ---------------------
        # Backpropagation
        # ---------------------

        while node is not None:

            node.visits += 1

            if result == 'X':
                node.wins += 1

            elif result == 'Draw':
                node.wins += 0.5

            node = node.parent

    # Best move = most visited child
    best_child = max(
        root.children,
        key=lambda child: child.visits
    )

    return best_child.move


# =========================================
# TEST CASE 1
# =========================================

print("TEST CASE 1")

game1 = TicTacToe()

game1.print_board()

move1 = mcts(game1)

print("Best Move:", move1)

print()


# =========================================
# TEST CASE 2
# X CAN WIN
# =========================================

print("TEST CASE 2")

game2 = TicTacToe()

game2.board = [
    ['X', 'X', ' '],
    ['O', 'O', ' '],
    [' ', ' ', ' ']
]

game2.print_board()

move2 = mcts(game2)

print("Best Move:", move2)

print()


# =========================================
# TEST CASE 3
# BLOCK O
# =========================================

print("TEST CASE 3")

game3 = TicTacToe()

game3.board = [
    ['O', 'O', ' '],
    ['X', ' ', ' '],
    ['X', ' ', ' ']
]

game3.print_board()

move3 = mcts(game3)

print("Best Move:", move3)

print()


# =========================================
# TEST CASE 4
# DRAW POSITION
# =========================================

print("TEST CASE 4")

game4 = TicTacToe()

game4.board = [
    ['X', 'O', 'X'],
    ['X', 'O', 'O'],
    ['O', 'X', ' ']
]

game4.print_board()

move4 = mcts(game4)

print("Best Move:", move4)

TEST CASE 1
['X', 'O', 'X']
['O', 'X', ' ']
[' ', ' ', 'O']
Best Move: (2, 0)

TEST CASE 2
['X', 'X', ' ']
['O', 'O', ' ']
[' ', ' ', ' ']
Best Move: (0, 2)

TEST CASE 3
['O', 'O', ' ']
['X', ' ', ' ']
['X', ' ', ' ']
Best Move: (0, 2)

TEST CASE 4
['X', 'O', 'X']
['X', 'O', 'O']
['O', 'X', ' ']
Best Move: (2, 2)
